# Intro to Python in Chem 432 Part V

This lesson and subsequent ones aim to teach you a bit of Python coding and data visualization. This workbook builds on Parts I - IV.

A few things to start:

1.   These lessons only work in Google Chrome
2.   Save a copy in your google drive. Go to File> Save a Copy in Drive; then locate a spot in your Drive folder
3.   You can save these notebooks and run offline as a Jupyter Notebook.
4.   When you finish with the assignment, either **download it as an .ipynb** and submit the notebook to canvas, or **share it with me** and give me edit access.

If you have questions, feel free to contact Dr. Sumner.

##Ro-vibrational Spectra

In this module, we will use Python/Colab to read in and analyze ro-vibrational spectroscpy data pulled from an FT-IR of carbon monoxide. This data is in a CSV file and taken directly from the instrument. At the end of the module, you will have calculated the spectroscopic constants, $\omega_e$, $\omega_e\chi_e$, $B_e$, $\alpha_e$, and $D_e$. If you haven't done so already, please upload the data to your google drive: https://raw.githubusercontent.com/Sumner-Group/Python_for_Physical_Chemistry/main/student_notebooks/data/101723_CO-IRData-From2023438L.CSV

##Importing your Google Drive
Since Colab is connected to your Google Drive, it can read and write files to your drive. This is convenient. Unfortunately, it will not do this by default, so we need to give Colab permission to read and write files in your drive, hence the **code chunk** below. Google will probably send you a lot of warnings. You must click "OK" on them all.


In [ ]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)


In [ ]:
# Add in a markdown title. This one should not rerun automatically
#@title **Please, provide the necessary input files below:**
# Add in a markdown paramter, filename. Make sure its type is string.
# (String basically means characters or not numbers)
filename = '101723_CO-IRData-From2023438L.CSV' #@param {type:"string"}
#Add in a markdown paramter, Google_Drive_Path.
# This should start with /content/drive/My Drive/
#It should end with whatever folders and subfolders you use to organize your google drive.
#This should also be type, string
Google_Drive_Path = '/content/drive/My Drive/Colab Notebooks/' #@param {type:"string"}
#Create a new string that is the Google_Drive_Path plus the filename
csv=Google_Drive_Path+filename


There is an alternate way to access data, if you don't want to use Google Drive. It requires the use of a Python library called "Pandas". You could use the following code:


```
import pandas as pd
csv=pd.read_csv('https://raw.githubusercontent.com/Sumner-Group/Python_for_Physical_Chemistry/main/student_notebooks/data/101723_CO-IRData-From2023438L.CSV ',header=None)
```
The above code will directly download the data and not store it on your Drive. Furthermore, if you use this version, the line of code seen later in the notebook
```
nu,Iabs=np.genfromtxt(csv,delimiter=',',unpack=True)
```
won't work. You will need to replace it with
```
nu=csv.iloc[:,0]
Iabs=csv.iloc[:,1]
```
or


```
nu,Iabs=csv.iloc[:,[0,1]].T.values
```

Pandas is super useful. But Colab can interface with your Google Drive, which I think is cool. So that's the route we will go.

Next, we load in the necessary libraries. Remember, if there is a blank line under a comment, you need to add in the code.

In [ ]:
# Load in NumPy
import numpy as np
# Load in MatPlotLib
import matplotlib.pyplot as plt
# Load in a specific module from SciPy. This module will allow us to locate peaks in a spectrum
from scipy.signal import find_peaks

Now, we are going to use an NumPy function that will read in the CSV file. The CSV file has two columns. The first column contains the wavenumber and the second column contains the contains the intensity of the peak. We are going to read the fie in using the following command:

`nu,Iabs=np.genfromtxt(csv,delimiter=',',unpack=True)`

The part that reads `np.genfromtxt(csv,delimiter=',',unpack=True)` means that we are going to read in a file that we call `csv` (see previous **code chunk**). The columns are marked with commas `delimiter=','` (CSV means comma separated values). `unpack=True` meas that we need to turn the vertical arrays we see in the CSV file to horizontal arrays that python likes by default.

Notice that when we read this file in, we are giving it two variable names, `nu` and `Iabs`. This means that the first column (peak positions in wavenumbers) will go into an array we call `nu` and the second column (peak intensities) will go into an array called `Iabs`

In [ ]:
#read in the csv file and put the columns into two arrays called nu and Iabs
nu,Iabs=np.genfromtxt(csv,delimiter=',',unpack=True)

# plot the spectrum using nu as the x-axis and Iabs as the y-axis
plt.plot(nu,Iabs,lw=1)
plt.show()

Once you've plotted the specta you should see a problem. There are many peaks that you don't need. We are only looking for the ro-vibrational peaks that exist between about 2000cm$^{-1}$ to 2230 cm$^{-1}$. So, we need to loop through all elements in the `nu` and `Iabs` arrays and append those elements that are in the 2000cm$^{-1}$ $-$ 2230 cm$^{-1}$ range to new arrays, `nu_v01` and `Iabs_v01.` A few hints that may help:

`len(nu)` gives you the number of elements in `nu`, so you can loop thorugh `range(len(nu))` to loop through all the elements. In an `if` statement:

`>` means greater than

`<` means less than

`>=` means greater than or equal to

`<=` means less than or equal to

`!=` means not equal to

`==` means equal

`and` means and

`or` means or


In [ ]:
#create two arrays, nu_v01 and Iabs_v01
nu_v01=[]
Iabs_v01=[]
#loop through all elements of nu
for i in range(len(nu)):
  #if the value nu[i] is between 2000 and 2230
  if(nu[i]>2000 and nu[i]<2230):
    #append the value of nu[i] to nu_v01
    nu_v01.append(nu[i])
    #append the value of Iabs[i] to Iabs_v01
    Iabs_v01.append(Iabs[i])
#Turn the arrays to numpy arrays
nu_v01=np.array(nu_v01)
Iabs_v01=np.array(Iabs_v01)
#plot the spectra. nu_v01 is on the x-axis and Iabs_v01 is on the y-axis
plt.plot(nu_v01,Iabs_v01,lw=1)
plt.show()

##Finding Peaks
Up to this point, we have only read in and "cleaned" data. The spectrum that we are interested in analyzing is stored in the arrays `nu_v01` and `Iabs_v01` The first thing we need to be able to do is find the peaks in the spectrum. This function exists in the library we loaded already: `from scipy.signal import find_peaks` We will use it like this:

`idx, _ = find_peaks(Iabs_v01, height=0.0)`

The function `find_peaks` takes two areguments. The first is the y-axis values, `Iabs_V01`, the second is the threshold above which we consider a peak. It is set to `0.0` in the above example. It puts the index of the array that has the peak in a new array called `idx`

After we find the peaks, we will check to make sure our code worked the way we wanted it to.

Be sure to play around with the cutoff value. Pick a values that seems to locate most of the peaks. **It is critical that the threshold is low enough that the $R\left(0\right)$ and $P\left(1\right)$ are located.**

In [ ]:
# Add in a markdown title that automatically reruns the code when new parameters are entered
# @title #Locate peaks { run: "auto" }
# Add in a markdown parameter, thresh, for the peak cutoff value
#@markdown Peak Height Cutoff
thresh=-0.0#@param{type:"number"}
# Find peaks
idx, _ = find_peaks(Iabs_v01, height=thresh)
# copy the values of nu_v01 that correspond to the peaks to an array called nu_peaks_v01
nu_peaks_v01 = nu_v01[idx]
# copy the values of Iabs_v01 that correspond to the peaks to an array called Iabs_peaks_v01
I_peaks_v01 = Iabs_v01[idx]
# Plot both the peaks and the spectrum. (nu_v01 vs Iabs_v01 & nu_peaks_v01 vs I_peaks_v01)
plt.plot(nu_v01, Iabs_v01, lw=1)
#plot nu_peaks as scatter, not as line, so the "picked" peaks will show up as dots.
plt.scatter(nu_peaks_v01, I_peaks_v01)
plt.show()

In the next **code chunk**, a function is defined that will number the P and R branches. A function takes arguments and returns a value. To define a function, use the following syntax: `def name_of_function(arguments, for, function):` The function below is called `assign_J` and it takes an array with the wavenumbers of the peaks `nu_peaks` as an argument, If you scoll down to the end of the function, it returns two values, `Jpp` and `Jp`, which are the rotational quantum numbers, $J''$ and $J'$, ie the lower and upper rotational quantum numbers.

This function will first look for the Q-branch by calculating the difference between the peaks. Once it finds the biggest gap, it knows that the P-branch is to the left and the R branch is to the right.

**You should ask Chat-GPT to explain this code to you.** ChatGPT is very good at reading/writing/debugging code. You can copy and paste this entire code chunk into ChatGPT and ask it to explain it to you. You can also copy and paste individial lines or functions like `np.arange(idx+1, 0, -1, dtype=int)` that you may not understand.

In [ ]:
def assign_J(nu_peaks):
    """
    Return arrays Jpp and Jp: rotational quantum numbers for the lower and upper
    states respectively for the vibrational transitions located at the
    wavenumbers nu_peaks.

    """

    npeaks = len(nu_peaks)
    print(npeaks, 'peaks provided to assign')

    # The spacing between consecutive elements in nu_peaks:
    dnu = np.diff(nu_peaks)
    # Find the index of the maximum spacing, corresponding to the band centre.
    idx = dnu.argmax()

    # We will store the rotational quantum numbers in these arrays.
    Jpp = np.empty(npeaks, dtype=int)
    Jp = np.empty(npeaks, dtype=int)

    # P-branch lines: everything up to (and including) idx,
    # decreasing with increasing index (and hence wavenumber).
    Jpp[:idx+1] = np.arange(idx+1, 0, -1, dtype=int)
    Jp[:idx+1] = Jpp[:idx+1] - 1      # P-branch: J' = J"-1
    # R-branch lines: everything from idx to the end of the array,
    # increasing with increasing index (and hence wavenumber).
    n = npeaks - idx - 1
    Jpp[idx+1:] = np.arange(0, n, 1)
    Jp[idx+1:] = Jpp[idx+1:] + 1      # R-branch: J' = J"+1

    return Jpp, Jp

Once the function is defined, we can use it find and label the P and R-branches. We do this by feeding the function the correct argument, `nu_peaks_v01`. We also want to make sure that we assign the outputs of the function to the proper variables. The outputs are arrays with the $J''$ and $J'$ rotational quantum numbers. We will call them `Jpp_v01` and `Jp_v01`. Since there are two variables, we set them both equal to the function in the correct order. The function puts $J''$ first and $J'$ second. So, add the following to the **code chunk**.

`Jpp_v01, Jp_v01 = assign_J(nu_peaks_v01)`

In [ ]:
#Label the P and R-branches
Jpp_v01, Jp_v01 = assign_J(nu_peaks_v01)

The next **code chunk** below will print out a small region of the spectrum with the peaks labeled so you can double check that you are on the right track. Again, **it is critical that the threshold is low enough that the $R\left(0\right)$ and $P\left(1\right)$ are correctly labeled.** If these peaks are not correctly labeled, lower your threshold.

In [ ]:
plt.plot(nu_v01, Iabs_v01)
xmin, xmax = 2120, 2165
plt.xlim(xmin, xmax)
plt.scatter(nu_peaks_v01, I_peaks_v01)
idx = np.where((xmin < nu_peaks_v01) & (nu_peaks_v01 < xmax))[0]
for i in idx:
    branch = 'P' if Jpp_v01[i] > Jp_v01[i] else 'R'
    # Label the peak with P(J") or R(J") with a line between the label
    # and the peak minimum. shrinkB backs off the line by 10 pts so it doesn't
    # actually connect with the peak: this looks better.
    plt.annotate(f'{branch}({Jpp_v01[i]})', xytext=(nu_peaks_v01[i], +0.06),
                 xy=(nu_peaks_v01[i], I_peaks_v01[i]), ha='center',
                 arrowprops={'arrowstyle': '-', 'shrinkB': 10})
plt.ylim(-0.02,0.05)
plt.xlabel(r'$\tilde{\nu}/cm^{-1}$')
plt.ylabel('$I_{abs}/arb. units$')
plt.show()

##Linear Regression and ro-vibrational spectroscopy##


 Let's examine the following equations:

 The full ro-vibrational energy equation (including anharmonicity, ro-vibrational coupling, and centrifugal distortion) is:

$\frac{E_{\nu,J}}{hc 100}=\omega_e(\nu+\frac{1}{2})-\omega_e\chi_e(\nu+\frac{1}{2})^2+B_eJ(J+1)-\alpha_e(\nu+\frac{1}{2})J(J+1)-D_eJ^2(J+1)^2$

Using this equation, we can calcuate the difference between the $\nu=0$ and $\nu=1$ energy levels. Remember that $\Delta J=J'-J''=\pm1$

$\frac{\Delta E}{hc 100}=\Delta\tilde{\nu}= \omega_e-2\omega_e\chi_e+B_e\left[J'(J'+1)-J''(J''+1)\right]+\alpha_e\left[\frac{1}{2}J''(J''+1)-\frac{3}{2}J'(J'+1)\right]+D_e\left[J''^2(J''+1)^2-J'^2(J'+1)^2\right]$

In the above equation, we know $\Delta\tilde{\nu}$ - the values of the transitions in wavenumbers pulled from experimental data - and the $J''$ and $J'$ numbers, because we labeled the $P$ and $R$ branches of the spectrum. We  want to find the values of $\omega_e-2\omega_e\chi_e$, $B_e$, $\alpha_e$, and $D_e$ the wil give us the best match to experiment.

This equation can be rewritten as a matrix-vector equation:

${\bf A}x=b$,

where $b$ is a vector (array) containing the values of $\Delta\tilde{\nu}$. $x$ is the vector:

\begin{equation}
x=\begin{pmatrix}
\omega_e-2\omega_e\chi_e \\
B_e \\
\alpha_e \\
D_e
\end{pmatrix}
\\
\end{equation}

Finally, ${\bf A}$ is a matrix, each row looks like:

\begin{equation}
{\bf A}_{n,\cdot} = \begin{pmatrix}
1, \left[J'(J'+1)-J''(J''+1)\right], \left[\frac{1}{2}J''(J''+1)-\frac{3}{2}J'(J'+1)\right],\left[J''^2(J''+1)^2-J'^2(J'+1)^2\right]
\end{pmatrix}
\end{equation}
 where $J''$ and $J'$ correspond to the $n^{th}$ transition in the $b$ ($\Delta\tilde{\nu}$) a vector. For example, if the $n^{th}$ element is the $P(3)$ line, then $J''=3$ and $J'=2$. If the  $n^{th}$ element is the $R(3)$ line, then then $J''=3$ and $J'=4$.

 Suppose we only have two lines in our spectrum, $P(1)$ and $R(0)$, the matrix-vector equation would look like this:

 \begin{equation}
 \begin{pmatrix}1 & -2 & 1 &4\\1 & 2 & -3 & -4\\\end{pmatrix}\begin{pmatrix}\omega_e-2\omega_e\chi_e\\B_e\\\alpha_e\\D_e\end{pmatrix}=\begin{pmatrix}P(1)\\R(0)\end{pmatrix}
 \end{equation}

 **You should review your matrix-vector multiplication rules and show that this equation is correct.**

 We want to find the set of parameters in $x$,
  \begin{equation}
x=\begin{pmatrix}\omega_e-2\omega_e\chi_e\\B_e\\\alpha_e\\D_e\end{pmatrix}
 \end{equation}

 that best fit the data, ie, make the following equation as close to zero as possible: $\|{\bf A}x-b\|_2=\sqrt{\left({\bf A}x-b\right)\cdot\left({\bf A}x-b\right)}$, where $\cdot$ means dot product.

 In the following **code chunk**, we will define a function that builds the $A$ matrix.

In [ ]:
#The inputs of this function are the J'' and J' arrays
def make_A(Jpp, Jp):
  #Create an empty Nx4 array, where N is the number of peaks you have, ie the
  #length of the Jpp array
  A=np.empty([len(Jpp),4])
  #Set the values of the first column of A - A[:,0] -  to 1
  A[:,0] = 1
  #Set the values of the second column of A to J'(J'+1)-J''(J''+1)
  A[:,1] = (Jp*(Jp+1) - Jpp*(Jpp+1))
  #Set the values of the third column of A to 1/2*J''*(J''+1) - 3/2*J'*(J'+1)
  A[:,2] = ((0.5)*Jpp*(Jpp+1) - (1.5)*Jp*(Jp+1))
  #Set the values of the fourth column of A to (J''*(J''+1))**2 - (J'*(J'+1))**2
  A[:,3] = ((Jpp*(Jpp+1))**2 - (Jp*(Jp+1))**2)
  #Return the A matrix
  return A



Finally, we will make the A matrix, and use the numpy function, `np.linalg.lstsq(A, nu_peaks_v01, rcond=None)` to find the least squares fit of the data, ie the set of parameters   $\begin{pmatrix}\omega_e-2\omega_e\chi_e\\B_e\\\alpha_e\\D_e\end{pmatrix}$ that best fit the data.
.

In [ ]:
#Construct the A matrix
A = make_A(Jpp_v01, Jp_v01)
#Run the linear least squares fit.
results = np.linalg.lstsq(A, nu_peaks_v01, rcond=None)
#The r-vib parameters are stored in the results array as the first row
prm_vals = results[0]
#
we_2wexe=prm_vals[0]
Be=prm_vals[1]
alphae=prm_vals[2]
De=prm_vals[3]
#Print the values of the parameters
print('The fit parameters (cm-1):')
print("we - 2wexe =",we_2wexe)
print("Be =",Be)
print("alphae =",alphae)
print("De =",De)


$B_e$, $\alpha_e$, and $D_e$ should be in good agreement with the literature values. The one major issue with this fit is that it gives you a value for $\omega_e-2\omega_e\chi_e$, but there isn't enough information to get $\omega_e\chi_e$ or $\omega_e$ by itself. If we want to find $\omega_e\chi_e$, we either need an overtone or a hotband (recall how we found these data for SiO in a previous homework set). However, there are a few approximations we can use. One approximation is due to Morse of potential fame (https://journals.aps.org/pr/abstract/10.1103/PhysRev.34.57), the other is due to Pekeris (https://journals.aps.org/pr/abstract/10.1103/PhysRev.45.98).

According to Morse:
\begin{equation}
\omega_e\chi_e\approx\frac{\alpha_e\omega_e}{2B_e}
\end{equation}
Since, we know that
\begin{equation}
\omega_e-2\omega_e\chi_e=b_0,
\end{equation}
where $b_0=$prm_vals[0], ie the constant from our least-squares fit. We can combine these equations to get
\begin{equation}
\omega_e\approx\frac{b_0}{1-\frac{\alpha_e}{B_e}}
\end{equation}

The equation from Pekeris is....gross:
\begin{equation}
\omega_e\chi_e\approx B_e\left(\frac{\alpha_e\omega_e}{6B_e^2}+1\right)^2
\end{equation}
Again, we can combine this with $\omega_e-2\omega_e\chi_e=b_0$, to get (and I am truly sorry),
\begin{equation}
\omega_e\approx\frac{-(\frac{2\alpha_e}{3B_e}-1)\pm \sqrt{\left(\frac{2\alpha_e}{3B_e}-1\right)^2-4\left(\frac{\alpha_e^2}{18Be^3}\right)\left(2B_e+b_0\right)}}{2\left(\frac{\alpha_e^2}{18Be^3}\right)}
\end{equation}
You can probably guess which one is more accurate.

Finally, there is one more equation we can use. The equation $\omega_e\chi_e\approx\frac{\alpha_e\omega_e}{2B_e}$, when compared with experiment, is shown to produce a value of $\omega_e\chi_e$ that is too low by a factor of about 1.4. So, we can account for this by multiplying $\omega_e\chi_e$ by 1.4 and we get the final equation:
\begin{equation}
\omega_e\approx\frac{b_0}{1-1.4\frac{\alpha_e}{B_e}}
\end{equation}

In the **code chunk** below, pick two of these formulas to calculate $\omega_e$ and $\omega_e\chi_e$ You may also recall from your HW that the Dissociation energy (using the Morse potential) is  $D_0=\frac{\omega_e^2}{4\omega_e\chi_e}-\frac{\omega_e}{2}+\frac{\omega_e\chi_e}{4}$. So calculate, $D_0$, too. Finally, since we know the rotational constant, we can also calculate the bond length of CO. Do it.

In [ ]:
#calculate and print out the values of we, wexe in two different ways
a=prm_vals[2]**2/(18*prm_vals[1]**3)
b=2*prm_vals[2]/(3*prm_vals[1])-1
c=2*prm_vals[1]+prm_vals[0]
we_pek=(-b-np.sqrt(b**2-4*a*c))/(2*a)
we_morse=prm_vals[0]/(1-prm_vals[2]/prm_vals[1])
we_morse2=prm_vals[0]/(1-1.4*prm_vals[2]/prm_vals[1])
wexe_pek=(we_pek-we_2wexe)/2
wexe_morse=(we_morse-we_2wexe)/2
wexe_morse2=(we_morse2-we_2wexe)/2
print("Morse")
print("we = ",we_morse, " wexe = ",wexe_morse,"\n")
print("Corrected Morse")
print("we = ",we_morse2, " wexe = ",wexe_morse2,"\n")
print("Pekeris")
print("we = ",we_pek, " wexe = ",wexe_pek)

In [ ]:
!pip install mendeleev

In [ ]:
#Use the most accurate values of we and wexe to calculate D0 and use Be to calculate
# the bond length of CO. print the values
from scipy import constants
from mendeleev import isotope
o_mass=isotope(8,16).mass
c_mass=isotope(6,12).mass
h=constants.Planck
c=constants.c
#
D_pek=we_pek**2/(4*wexe_pek)-we_pek/2+wexe_pek/4
D_morse=we_morse**2/(4*wexe_morse)-we_morse/2+wexe_morse/4
D_morse2=we_morse2**2/(4*wexe_morse2)-we_morse2/2+wexe_morse2/4
print("Bond dissociation Energy (Morse) in wavenumbers ",D_morse)
print("Bond dissociation Energy (Morse corrected) in wavenumbers ",D_morse2)
print("Bond dissociation Energy (Pekeris) in wavenumbers ",D_pek,"\n")
#
mu=o_mass*c_mass/(o_mass+c_mass)/(1000*constants.N_A)
r=np.sqrt(h/(8*np.pi**2*mu*Be*c*100))
print("Bond Length is ",r/1E-10, "Angstroms")


Adapted from the following source: https://scipython.com/chem/book/fitting-the-vibrational-spectrum-of-co/fitting-the-vibrational-spectrum-of-co/, which is a chapter from *Python for Chemists* by Christian Hill